<a href="https://colab.research.google.com/github/DeepLabCut/DeepLabCut/blob/master/examples/COLAB/COLAB_3miceDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 中打开"/></a>

# DeepLabCut 多小鼠数据演示
![alt text](https://images.squarespace-cdn.com/content/v1/57f6d51c9f74566f55ecf271/1628250004229-KVYD7JJVHYEFDJ32L9VJ/DLClogo2021.jpg?format=1000w)

https://github.com/DeepLabCut/DeepLabCut

注意：本 Colab 笔记本旨在配合发表在《自然-方法》（Nature Methods）上的论文《[Multi-animal pose estimation, identification and tracking with DeepLabCut](https://www.nature.com/articles/s41592-022-01443-0)》（使用 DeepLabCut 进行多动物姿态估计、识别和跟踪），它使用的是 TensorFlow 引擎。要了解 DeepLabCut 3.0+ 和 PyTorch 引擎，您可以查看我们的其他笔记本（例如 [`COLAB_YOURDATA_maDLC_TrainNetwork_VideoAnalysis.ipynb`](https://github.com/DeepLabCut/DeepLabCut/blob/main/examples/COLAB/COLAB_YOURDATA_maDLC_TrainNetwork_VideoAnalysis.ipynb)）。

## 本笔记本演示了如何使用 COLAB 进行多动物 DeepLabCut (maDLC) 演示项目（涉及 3 只小鼠）：

- 加载包含预训练模型和未标记视频的迷你演示数据。
- 分析新的视频。
- 组合动物身份和轨迹片段（tracklets）。
- 创建质量检查图表和视频。

- 要创建完整的 maDLC 流程（pipeline），请参阅我们的完整文档：https://deeplabcut.github.io/DeepLabCut/README.html

- 以下资源对 maDLC 特别有用：
    - maDLC 完整操作指南：https://deeplabcut.github.io/DeepLabCut/docs/maDLC_UserGuide.html
    - maDLC 快速指南：https://deeplabcut.github.io/DeepLabCut/docs/tutorial.html
    - 一个演示 COLAB 笔记本，展示如何将 maDLC 用于您自己的数据：https://github.com/DeepLabCut/DeepLabCut/blob/main/examples/COLAB/COLAB_maDLC_TrainNetwork_VideoAnalysis.ipynb

### 要开始使用，请转到 "Runtime" -> "change runtime type" -> 选择 "Python3"，然后选择 "GPU"

In [ ]:
# Install the correct (older) version of DeepLabCut
!pip install "deeplabcut[tf]"

## 重要提示 - 请重启运行时环境，以便导入已更新的包！

**请务必**在继续操作之前，点击上方输出中的 “Restart runtime”（重启运行时）按钮！

## 下载我们的演示项目到您的服务器：

No information needs edited in the cells below, you can simply click run on each:

## 下载我们的演示项目到您的服务器：

In [ ]:
# Download our demo project:
import requests
from io import BytesIO
from zipfile import ZipFile

url_record = 'https://zenodo.org/api/records/7883589'
response = requests.get(url_record)
if response.status_code == 200:
    file = response.json()['files'][0]
    title = file['key']
    print(f"Downloading {title}...")
    with requests.get(file['links']['self'], stream=True) as r:
        with ZipFile(BytesIO(r.content)) as zf:
            zf.extractall(path='/content')
else:
    raise ValueError(f'The URL {url_record} could not be reached.')

## 使用我们的 maDLC DLCRNet 分析一个新颖的 3 只小鼠视频（该网络已在 3 只小鼠数据集上预训练）：（此处您将提取检测结果和关联成本）

In [ ]:
import deeplabcut as dlc
import os

project_path = "/content/demo-me-2021-07-14"
config_path = os.path.join(project_path, "config.yaml")
video = os.path.join(project_path, "videos", "videocompressed1.mp4")

dlc.analyze_videos(config_path,[video], shuffle=0, videotype="mp4",auto_track=False )

## 接下来，您将计算局部的、时空分组，并逐帧跟踪身体部位的组件：

In [ ]:
TRACK_METHOD = "ellipse"  # Could also be "box", but "ellipse" was found to be more robust on this dataset.

dlc.convert_detections2tracklets(
    config_path,
    [video],
    videotype='mp4',
    shuffle=0,
    track_method=TRACK_METHOD,
    ignore_bodyparts=["tail1", "tail2", "tailend"],  # Some body parts can optionally be ignored during tracking for better assembly (but they are used later)
)

## 重建完整的动物轨迹（由轨迹片段组成的完整路径）：

In [ ]:
dlc.stitch_tracklets(
    config_path,
    [video],
    videotype='mp4',
    shuffle=0,
    track_method=TRACK_METHOD,
    n_tracks=3,
)

## 创建美观的视频输出：

In [ ]:
#Filter the predictions to remove small jitter, if desired:
dlc.filterpredictions(config_path, 
                                 [video], 
                                 shuffle=0,
                                 videotype='mp4', 
                                 track_method = TRACK_METHOD)

dlc.create_labeled_video(
    config_path,
    [video],
    videotype='mp4',
    shuffle=0,
    color_by="individual",
    keypoints_only=False,
    draw_skeleton=True,
    filtered=True,
    track_method=TRACK_METHOD,
)

现在，在左侧面板中，如果您点击文件夹图标，您将看到项目文件夹 `"demo-me.."`；点击进入该文件夹，然后进入 `"videos"` 目录，您就可以找到 `"..._id_labeled.mp4"` 视频文件，双击该文件即可下载并进行检查！

## 创建数据的图表：

> 运行完毕后，你可以在 "videos" 和 "plot-poses" 文件夹中查看轨迹数据！ (有时你需要点击文件夹的刷新图标才能看到它们)。在这些文件夹中，例如，你可以查看 `plotmus1.png` 文件，以**可视化身体部位随时间变化的像素位置**。

In [ ]:
dlc.plot_trajectories(config_path, [video], shuffle=0,videotype='mp4', track_method=TRACK_METHOD)